In [ ]:
import os
import cv2
import pandas as pd
from pathlib import Path

# 1. THE "FORCE" LOCK
# This manually points Python to your project folder. 
# Using forward slashes (/) prevents the 'unterminated string literal' error.
BASE_DIR = Path(r"C:\projects\PYTHON\GA03\AgriSort_Project")

# 2. DERIVED PATHS
# These will now automatically point to C:/projects/PYTHON/AgriSort/...
RAW_DIR = BASE_DIR / "raw_data_2"
PROCESSED_DIR = BASE_DIR / "processed_images"
CSV_PATH = BASE_DIR / "metadata.csv"

# Configurations
IMG_SIZE = 100 
categories = {'bananas': 0, 'oranges': 1}
metadata = []

# Ensure directories exist where we want them
if not PROCESSED_DIR.exists():
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Checking for raw data...")

count = 1
for category, label in categories.items():
    folder_path = RAW_DIR / category
    
    if not folder_path.exists():
        print(f"Warning: Folder {folder_path} not found. Skipping...")
        continue

    # Get list of subfolders
    subfolders = [f.name for f in folder_path.iterdir() if f.is_dir()]

    for subfolder in subfolders:
        subfolder_path = folder_path / subfolder
        # Get list of images
        files = [f for f in os.listdir(subfolder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        for file in files:
            try:
                # Load
                img = cv2.imread(str(subfolder_path / file))
                
                if img is None:
                    continue
                    
                # Resize
                resized_img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
                
                # Save with 4-digit padding (0001.jpg, etc.)
                new_filename = f"{count:04d}.jpg"
                cv2.imwrite(str(PROCESSED_DIR / new_filename), resized_img)
                
                # Record
                metadata.append({
                    'id': count, 
                    'file_name': new_filename, 
                    'class': label
                })
                count += 1
                
            except Exception as e:
                print(f"Skipping {file} due to error: {e}")

# 3. VERIFIED SAVING
if metadata:
    df = pd.DataFrame(metadata)
    # We use the absolute path we defined at the top
    df.to_csv(str(CSV_PATH), index=False)
    print("Done")
else:
    print("No metadata generated. Check if raw_data contains images.")